In [2]:
from pyspark.sql.functions import (
    col, date_format, hour, dayofweek, weekofyear, month, year,
    monotonically_increasing_id, when, lag,
    sum as spark_sum, min as spark_min, max as spark_max, unix_timestamp
)
from pyspark.sql import Window

# Read from Silver
df_silver = spark.read.format("delta").load("abfss://TfL-Analytics@onelake.dfs.fabric.microsoft.com/TfL_Silver_LH.Lakehouse/Tables/silver_line_status")

# ===========================
# Dim_Line
# ===========================
dim_line = df_silver \
    .select("line_id", "line_name", "mode_name") \
    .distinct()

dim_line.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://TfL-Analytics@onelake.dfs.fabric.microsoft.com/TfL_Gold_LH.Lakehouse/Tables/dim_line")

print(f"✅ Dim_Line: {dim_line.count()} records")

# ===========================
# Dim_Date
# ===========================
dim_date = df_silver \
    .select("ingestion_timestamp") \
    .distinct() \
    .withColumn("date_id", date_format(col("ingestion_timestamp"), "yyyyMMddHHmm")) \
    .withColumn("date", date_format(col("ingestion_timestamp"), "yyyy-MM-dd")) \
    .withColumn("year", year(col("ingestion_timestamp"))) \
    .withColumn("month", month(col("ingestion_timestamp"))) \
    .withColumn("week", weekofyear(col("ingestion_timestamp"))) \
    .withColumn("day_of_week", dayofweek(col("ingestion_timestamp"))) \
    .withColumn("hour", hour(col("ingestion_timestamp"))) \
    .withColumn("is_weekend", when(dayofweek(col("ingestion_timestamp")).isin(1, 7), True).otherwise(False)) \
    .withColumn("time_of_day",
        when(hour(col("ingestion_timestamp")).between(6, 9), "Morning Peak")
        .when(hour(col("ingestion_timestamp")).between(10, 15), "Midday")
        .when(hour(col("ingestion_timestamp")).between(16, 19), "Evening Peak")
        .otherwise("Off Peak"))

dim_date.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://TfL-Analytics@onelake.dfs.fabric.microsoft.com/TfL_Gold_LH.Lakehouse/Tables/dim_date")

print(f"✅ Dim_Date: {dim_date.count()} records")

# ===========================
# Fact_Disruption
# ===========================
# The raw Silver data is polled every 15 minutes, so a single real-world
# disruption can appear as dozens of near-identical consecutive rows.
# To distinguish genuine events from polling noise, we use a window function
# to compare each row to the previous row for the same line and flag the
# start of a new event whenever the status or reason changes.

window_spec = Window.partitionBy("line_id").orderBy("ingestion_timestamp")

fact_disruption = df_silver \
    .withColumn("date_id", date_format(col("ingestion_timestamp"), "yyyyMMddHHmm")) \
    .withColumn("prev_status", lag("status_description").over(window_spec)) \
    .withColumn("prev_reason", lag("disruption_reason").over(window_spec)) \
    .withColumn(
        "is_new_event",
        when(
            (col("prev_status").isNull()) |
            (col("prev_status") != col("status_description")) |
            (col("prev_reason") != col("disruption_reason")),
            1
        ).otherwise(0)
    ) \
    .select(
        monotonically_increasing_id().alias("fact_id"),
        col("line_id"),
        col("date_id"),
        col("status_severity"),
        col("status_description"),
        col("disruption_reason"),
        col("disruption_from"),
        col("disruption_to"),
        col("is_active_disruption"),
        col("disruption_category"),
        col("closure_text"),
        col("is_new_event"),
        col("ingestion_timestamp")
    )

fact_disruption.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://TfL-Analytics@onelake.dfs.fabric.microsoft.com/TfL_Gold_LH.Lakehouse/Tables/fact_disruption")

print(f"✅ Fact_Disruption: {fact_disruption.count()} records")

# ===========================
# Event_Duration
# ===========================
# Building on is_new_event, we group consecutive polling rows that belong to
# the same real-world event using a running sum of is_new_event as a group ID.
# For each group we take the earliest and latest timestamp to derive how long
# that specific disruption actually lasted.

window_spec_duration = Window.partitionBy("line_id").orderBy("ingestion_timestamp")

df_with_groups = fact_disruption.withColumn(
    "event_group",
    spark_sum("is_new_event").over(window_spec_duration)
)

event_duration = df_with_groups.groupBy("line_id", "event_group") \
    .agg(
        spark_min("ingestion_timestamp").alias("event_start"),
        spark_max("ingestion_timestamp").alias("event_end"),
        spark_min("status_description").alias("status_description"),
        spark_min("disruption_reason").alias("disruption_reason")
    ) \
    .withColumn(
        "duration_minutes",
        (unix_timestamp("event_end") - unix_timestamp("event_start")) / 60
    ) \
    .withColumn(
        "date_id",
        date_format(col("event_start"), "yyyyMMddHHmm")
    ) \
    .join(dim_line.select("line_id", "line_name"), on="line_id", how="left")

event_duration.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://TfL-Analytics@onelake.dfs.fabric.microsoft.com/TfL_Gold_LH.Lakehouse/Tables/event_duration")

print(f"✅ Event_Duration: {event_duration.count()} records")
print("🎉 Gold Layer complete!")

StatementMeta(, e300a266-8fd8-4415-8448-f1721cd86169, 4, Finished, Available, Finished, False)

✅ Dim_Line: 11 records
✅ Dim_Date: 1059 records
✅ Fact_Disruption: 11649 records
✅ Event_Duration: 665 records
🎉 Gold Layer complete!


In [8]:
# Register Gold tables

spark.sql("""
    CREATE TABLE IF NOT EXISTS dim_line
    USING DELTA
    LOCATION 'abfss://TfL-Analytics@onelake.dfs.fabric.microsoft.com/TfL_Gold_LH.Lakehouse/Tables/dim_line'
""")

spark.sql("""
    CREATE TABLE IF NOT EXISTS dim_date
    USING DELTA
    LOCATION 'abfss://TfL-Analytics@onelake.dfs.fabric.microsoft.com/TfL_Gold_LH.Lakehouse/Tables/dim_date'
""")

spark.sql("""
    CREATE TABLE IF NOT EXISTS fact_disruption
    USING DELTA
    LOCATION 'abfss://TfL-Analytics@onelake.dfs.fabric.microsoft.com/TfL_Gold_LH.Lakehouse/Tables/fact_disruption'
""")

spark.sql("""
    CREATE TABLE IF NOT EXISTS event_duration
    USING DELTA
    LOCATION 'abfss://TfL-Analytics@onelake.dfs.fabric.microsoft.com/TfL_Gold_LH.Lakehouse/Tables/event_duration'
""")


spark.sql("""
    CREATE TABLE IF NOT EXISTS mtbd
    USING DELTA
    LOCATION 'abfss://TfL-Analytics@onelake.dfs.fabric.microsoft.com/TfL_Gold_LH.Lakehouse/Tables/mtbd'
""")
print("✅ MTBD table registered!")

print("✅ All Gold tables registered!")

StatementMeta(, e300a266-8fd8-4415-8448-f1721cd86169, 10, Finished, Available, Finished, False)

✅ MTBD table registered!
✅ All Gold tables registered!


In [9]:
from pyspark.sql.functions import col, when

# Read Gold dim_date
df_date = spark.read.format("delta").load("abfss://TfL-Analytics@onelake.dfs.fabric.microsoft.com/TfL_Gold_LH.Lakehouse/Tables/dim_date")

# Add Day Name column
df_date = df_date.withColumn("day_name",
    when(col("day_of_week") == 1, "Sunday")
    .when(col("day_of_week") == 2, "Monday")
    .when(col("day_of_week") == 3, "Tuesday")
    .when(col("day_of_week") == 4, "Wednesday")
    .when(col("day_of_week") == 5, "Thursday")
    .when(col("day_of_week") == 6, "Friday")
    .when(col("day_of_week") == 7, "Saturday")
)

# Save back to Gold
df_date.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://TfL-Analytics@onelake.dfs.fabric.microsoft.com/TfL_Gold_LH.Lakehouse/Tables/dim_date")

print("✅ Day Name column added!")

StatementMeta(, e300a266-8fd8-4415-8448-f1721cd86169, 11, Finished, Available, Finished, False)

✅ Day Name column added!


In [10]:
from pyspark.sql.functions import lead, col, unix_timestamp, min as spark_min, max as spark_max, sum as spark_sum, avg as spark_avg
from pyspark.sql import Window

# ===========================
# MTBD (Mean Time Between Disruptions)
# ===========================
# We exclude statuses that are not genuine, unexpected disruptions:
# Good Service, Planned Closure, Service Closed (scheduled), 
# Reduced Service and Special Service (planned/known in advance)

window_mtbd = Window.partitionBy("line_id").orderBy("event_start")

df_disruptions_only = event_duration.filter(
    ~col("status_description").isin(
        "Good Service", "Planned Closure", "Service Closed",
        "Reduced Service", "Special Service"
    )
)

# Find the next disruption start time for each event, per line
df_mtbd = df_disruptions_only.withColumn(
    "next_event_start",
    lead("event_start").over(window_mtbd)
)

# Calculate time between consecutive disruptions, in minutes
df_mtbd = df_mtbd.withColumn(
    "time_between_disruptions",
    (unix_timestamp("next_event_start") - unix_timestamp("event_start")) / 60
)

# Aggregate to one row per line: average, min, and max gap between disruptions
mtbd = df_mtbd.groupBy("line_id", "line_name") \
    .agg(
        spark_avg("time_between_disruptions").alias("mtbd_minutes"),
        spark_min("time_between_disruptions").alias("min_time_between"),
        spark_max("time_between_disruptions").alias("max_time_between")
    )

# Save to Gold layer
mtbd.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://TfL-Analytics@onelake.dfs.fabric.microsoft.com/TfL_Gold_LH.Lakehouse/Tables/mtbd")

print(f"✅ MTBD: {mtbd.count()} records")

StatementMeta(, e300a266-8fd8-4415-8448-f1721cd86169, 12, Finished, Available, Finished, False)

✅ MTBD: 11 records
